# TP02 — Mineração de Dados — Grupo 06
## Notebook 2/3 — Pré-processamento

Carrega a base **bruta**, define a população e o alvo do Problema B, faz a engenharia de atributos, o split estratificado 60/20/20 e o undersampling, e **exporta as bases tratadas** (em `./bases_tratadas/`) consumidas pelo notebook de Modelagem (3/3).

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time, os

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

FIG_DIR = "figs"
os.makedirs(FIG_DIR, exist_ok=True)
print("Setup OK")

Setup OK


**Decisão de carregamento.** A base é um arquivo de origem governamental com características específicas:
- separador de campos `|`;
- codificação **Latin-1** (não UTF-8) — contém acentuação (`Ç`, `Ã`...);
- **vírgula como separador decimal** (ex.: `622,5`). Sem `decimal=","`, as notas objetivas e com peso seriam lidas como texto.

In [4]:
CSV = "lista_de_espera_sisu_2023_2.csv"
df = pd.read_csv(CSV, sep="|", encoding="latin-1", decimal=",", low_memory=False)
print("Shape:", df.shape)
df.head(3)

Shape: (148773, 56)


,ANO,EDICAO,ETAPA,DS_ETAPA,CODIGO_IES,NOME_IES,SIGLA_IES,UF_IES,CODIGO_CAMPUS,NOME_CAMPUS,UF_CAMPUS,MUNICIPIO_CAMPUS,CODIGO_CURSO,NOME_CURSO,GRAU,TURNO,DS_PERIODICIDADE,TP_COTA,TIPO_MOD_CONCORRENCIA,MOD_CONCORRENCIA,QT_VAGAS_CONCORRENCIA,PERCENTUAL_BONUS,PESO_L,PESO_CH,PESO_CN,PESO_M,PESO_R,NOTA_MINIMA_L,NOTA_MINIMA_CH,NOTA_MINIMA_CN,NOTA_MINIMA_M,NOTA_MINIMA_R,MEDIA_MINIMA,CPF,INSCRICAO_ENEM,INSCRITO,SEXO,DT_NASCIMENTO,UF_CANDIDATO,MUNICIPIO_CANDIDATO,OPCAO,NOTA_L,NOTA_CH,NOTA_CN,NOTA_M,NOTA_R,NOTA_L_COM_PESO,NOTA_CH_COM_PESO,NOTA_CN_COM_PESO,NOTA_M_COM_PESO,NOTA_R_COM_PESO,NOTA_CANDIDATO,NOTA_CORTE,CLASSIFICACAO,APROVADO,MATRICULA
0,2023,2,7,LISTA DE ESPERA,593,CENTRO FEDERAL DE EDUCAÇÃO TECNOLÓGICA CELSO S...,CEFET/RJ,RJ,1663,CEFET-RJ - MARIA DA GRAÇA,RJ,Rio de Janeiro,1441998,SISTEMAS DE INFORMAÇÃO,Bacharelado,Noturno,Semestral,NaN,A,Ampla concorrência,15,NaN,1.0,1.0,2.0,4.0,3.0,453.8,444.7,453.3,438.4,300.0,418.04,XXX.398407-XX,221XXXXXX236,ALBERTH TEIXEIRA DA ROCHA,M,2001,RJ,Rio de Janeiro,2,620.3,630.2,553.2,731.6,660,620.3,630.2,1106.4,2926.4,1980,660.30,660.69,16,S,NÃO CONVOCADO
1,2023,2,7,LISTA DE ESPERA,593,CENTRO FEDERAL DE EDUCAÇÃO TECNOLÓGICA CELSO S...,CEFET/RJ,RJ,1663,CEFET-RJ - MARIA DA GRAÇA,RJ,Rio de Janeiro,1441998,SISTEMAS DE INFORMAÇÃO,Bacharelado,Noturno,Semestral,NaN,A,Ampla concorrência,15,NaN,1.0,1.0,2.0,4.0,3.0,453.8,444.7,453.3,438.4,300.0,418.04,XXX.787587-XX,221XXXXXX458,ALEXANDRE VIANA GONCALVES,M,1994,RJ,Rio de Janeiro,1,631.3,604.8,621.1,673.1,580,631.3,604.8,1242.2,2692.4,1740,628.25,660.69,27,S,NÃO CONVOCADO
2,2023,2,7,LISTA DE ESPERA,593,CENTRO FEDERAL DE EDUCAÇÃO TECNOLÓGICA CELSO S...,CEFET/RJ,RJ,1663,CEFET-RJ - MARIA DA GRAÇA,RJ,Rio de Janeiro,1441998,SISTEMAS DE INFORMAÇÃO,Bacharelado,Noturno,Semestral,NaN,A,Ampla concorrência,15,NaN,1.0,1.0,2.0,4.0,3.0,453.8,444.7,453.3,438.4,300.0,418.04,XXX.641477-XX,221XXXXXX526,ANA JULIA MELO DA SILVA,F,2004,RJ,Resende,2,534.8,550.8,468.5,612.0,820,534.8,550.8,937.0,2448.0,2460,630.05,660.69,26,S,NÃO CONVOCADO


### 1.3 População e variável-alvo do Problema B
A pergunta é condicionada a **ter sido convocado**. O campo `MATRICULA` codifica o desfecho:

| Status | Significado | Uso no Problema B |
|---|---|---|
| `NÃO CONVOCADO` | nunca foi chamado | **excluído** (fora da condição "foi convocado") |
| `EFETIVADA` | matrícula concluída | alvo **positivo (1)** |
| `NÃO COMPARECEU` | não se apresentou | negativo (0) |
| `DOCUMENTACAO REJEITADA` | documentação reprovada | negativo (0) |
| `CANCELADA` | matrícula cancelada | negativo (0) |
| `PENDENTE` | processo em andamento | **excluído** (desfecho ainda indefinido — rótulo ambíguo) |

**Decisão:** alvo binário `EFETIVOU` = 1 se `EFETIVADA`, 0 para os demais desfechos *resolvidos*; excluímos `PENDENTE` para não rotular como "não efetivou" casos cujo resultado ainda não existe.

In [5]:
print("MATRICULA — status na base completa:")
print(df["MATRICULA"].value_counts(dropna=False))

convocados = df[df["MATRICULA"] != "NÃO CONVOCADO"].copy()
base = convocados[convocados["MATRICULA"] != "PENDENTE"].copy()
base["EFETIVOU"] = (base["MATRICULA"] == "EFETIVADA").astype(int)

print("\nConvocados (excl. NÃO CONVOCADO):", len(convocados))
print("Resolvidos p/ modelagem (excl. PENDENTE):", len(base))
print("\nDistribuição do alvo EFETIVOU:")
print(base["EFETIVOU"].value_counts())
print("Taxa de efetivação: %.1f%%" % (100 * base["EFETIVOU"].mean()))

MATRICULA — status na base completa:
MATRICULA
NÃO CONVOCADO             69729
PENDENTE                  34540
NÃO COMPARECEU            26085
EFETIVADA                 16943
DOCUMENTACAO REJEITADA     1233
CANCELADA                   243
Name: count, dtype: int64

Convocados (excl. NÃO CONVOCADO): 79044
Resolvidos p/ modelagem (excl. PENDENTE): 44504

Distribuição do alvo EFETIVOU:
EFETIVOU
0    27561
1    16943
Name: count, dtype: int64
Taxa de efetivação: 38.1%


### 1.4 Engenharia de atributos para a EDA
Criamos três atributos derivados que serão usados na modelagem:
- **`MESMA_UF`** (obrigatória no enunciado): candidato concorre na mesma UF do campus (`UF_CANDIDATO == UF_CAMPUS`). -> Quem foi aprovado fora do estado tende a desistir. Na EDA: efetivação 43,7% na mesma UF vs 18,8% em outra UF.
- **`IDADE`**: `2023 - DT_NASCIMENTO` (o campo de nascimento é o ano). -> Quanto mais novo (recém saído do EM) maior a chance de efetivar
- **`MARGEM`**: `NOTA_CANDIDATO - NOTA_CORTE` (quão acima da nota de corte o candidato ficou). -> Se passou raspando, pode ter passado na opção menos desejada.

In [6]:
base["MESMA_UF"] = (base["UF_CANDIDATO"].astype(str) == base["UF_CAMPUS"].astype(str)).astype(int)
base["IDADE"]    = 2023 - base["DT_NASCIMENTO"]
base["MARGEM"]   = base["NOTA_CANDIDATO"] - base["NOTA_CORTE"]
base[["MESMA_UF", "IDADE", "MARGEM"]].describe().round(2)

,MESMA_UF,IDADE,MARGEM
count,44504.00,44504.00,41921.00
mean,0.77,21.44,-21.32
std,0.42,5.97,61.20
min,0.00,9.00,-425.38
25%,1.00,19.00,-51.10
50%,1.00,19.00,-7.27
75%,1.00,21.00,13.90
max,1.00,75.00,351.52


## Etapa 2 — Pré-processamento

Aplicamos as decisões dos checkpoints CP-A/CP-B. Pontos-chave para evitar **vazamento (leakage)**:
- a imputação por mediana e a normalização são **ajustadas apenas no treino** (dentro do `Pipeline`/`ColumnTransformer`);
- o **undersampling** é aplicado **somente ao conjunto de treino** — validação e teste permanecem com a distribuição real (38% positivos).

### 2.1 Seleção de features e tratamento de faltantes

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

num_features = ["NOTA_CANDIDATO", "NOTA_CORTE", "MARGEM", "CLASSIFICACAO", "IDADE", "QT_VAGAS_CONCORRENCIA"]
cat_features = ["GRAU", "TURNO", "TIPO_MOD_CONCORRENCIA", "TP_COTA", "SEXO", "OPCAO"]
bin_features = ["MESMA_UF"]
ALL_FEATURES = num_features + cat_features + bin_features

X = base[ALL_FEATURES].copy()
y = base["EFETIVOU"].copy()

# TP_COTA: NaN = ampla concorrência / sem cota específica -> categoria própria (preenchimento constante, sem leakage)

# AINDA SOBRE ISSO. Optei por preencher como ampla. Eu poderia fazer a moda, pq a moda percorreria a base de testes/valuation tb, isso poderia causar um leakege
X["TP_COTA"] = X["TP_COTA"].fillna("AMPLA")

print("X:", X.shape, "| y positivos: %.1f%%" % (100*y.mean()))
print("\nNaN restantes por coluna (serão imputados por mediana no pipeline):")
print(X.isna().sum()[X.isna().sum() > 0])

X: (44504, 13) | y positivos: 38.1%

NaN restantes por coluna (serão imputados por mediana no pipeline):
NOTA_CORTE    2583
MARGEM        2583
dtype: int64


### 2.2 Separação treino / validação / teste
Estratégia **Holdout** estratificada (preserva 38% de positivos em cada partição), `random_state=42`:
- **Teste = 20%** (avaliação final, intocado);
- dos 80% restantes: **Treino = 60%** e **Validação = 20%** (a validação é usada para a busca de hiperparâmetros).

In [8]:
# 1º corte: separa o teste (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
# 2º corte: dos 80% restantes, 25% viram validação (=20% do total) e 75% treino (=60% do total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=RANDOM_STATE)

for nome, yy in [("Treino", y_train), ("Validação", y_val), ("Teste", y_test)]:
    print(f"{nome:>10}: {len(yy):6d} linhas | positivos: {100*yy.mean():.1f}%")

    Treino:  26702 linhas | positivos: 38.1%
 Validação:   8901 linhas | positivos: 38.1%
     Teste:   8901 linhas | positivos: 38.1%


### 2.3 Tratamento do desbalanceamento — Undersampling (apenas no treino)

O treino tem ~38% de positivos. Reduzimos a classe majoritária (*não efetivou*) ao tamanho da minoria, **somente no conjunto de treino** — validação e teste mantêm a distribuição real (38%), pois é nessa distribuição que o modelo será avaliado e operará. Balancear val/teste mediria o desempenho num mundo 50/50 que não existe.

Escolhemos **undersampling** (em vez de `class_weight` ou oversampling/SMOTE) por dois motivos:

**1. Funciona igual para os 6 modelos.** O undersampling atua sobre os **dados**, reequilibrando o próprio conjunto de treino — é uma operação **independente do modelo**. Já o `class_weight` atua na **função de custo interna** do estimador e **só existe em alguns** deles: Árvore/Random Forest/SVM o suportam, mas **KNN** (decide por vizinhança/distância, não tem função de custo ponderável) e o **GaussianNB** (estima as probabilidades direto dos dados) **não têm** como ponderar classes. Ao balancear os dados *a montante*, os 6 modelos recebem exatamente o mesmo treino 50/50, sem precisar de um truque diferente para cada um — o que mantém a **comparação justa** (mesma distribuição de treino para todos).

**2. Acelera o SVM.** O custo de treino do SVM com kernel (`SVC`) cresce de forma **super-linear** com o número de amostras `n`: a matriz de kernel é `n × n` e a otimização quadrática escala aproximadamente entre `O(n²)` e `O(n³)`. Por isso, cortar a classe majoritária (de 26.702 → 20.330 linhas) reduz o tempo **mais do que proporcionalmente** à fração de linhas removidas, e ainda tende a produzir menos vetores de suporte. Para os outros 5 modelos a economia de tempo é só um bônus; para o SVM é o que torna a busca de hiperparâmetros viável (ele é o mais caro da etapa de tuning).

Custo aceito: descartamos ~6,4 mil negativos reais — tolerável dado o tamanho da base, e validado pelo bom AUC-ROC final.

In [9]:
treino = X_train.copy()
treino["EFETIVOU"] = y_train.values
pos = treino[treino["EFETIVOU"] == 1]
neg = treino[treino["EFETIVOU"] == 0]
n = min(len(pos), len(neg))

bal = pd.concat([pos.sample(n, random_state=RANDOM_STATE),
                 neg.sample(n, random_state=RANDOM_STATE)]
                ).sample(frac=1, random_state=RANDOM_STATE)  # embaralha
X_train_bal = bal.drop(columns="EFETIVOU")
y_train_bal = bal["EFETIVOU"]

print("Treino antes :", dict(y_train.value_counts()))
print("Treino depois:", dict(y_train_bal.value_counts()), "(balanceado)")

Treino antes : {0: 16537, 1: 10165}
Treino depois: {1: 10165, 0: 10165} (balanceado)


### 2.4 Pipeline de pré-processamento
Um único `ColumnTransformer`, reutilizado pelos 6 modelos (garante comparação justa):
- **numéricas** → `SimpleImputer(mediana)` + **`RobustScaler`** (decisão CP-A: robusto a outliers; normalização é obrigatória p/ KNN e SVM e inofensiva p/ árvores);
- **categóricas nominais** → **`OneHotEncoder`** (decisão CP-B);
- **`MESMA_UF`** já é binária → repassada direto.

#### Decisões

1. SimpleImputer(strategy="median") para NOTA_CORTE/MARGEM (2.583 NaN cada):
- Mediana e não média → robusta a outliers (a MARGEM tem cauda longa: vai de −425 a +351). A média seria puxada pelos extremos.
- Dentro do pipeline → a mediana é aprendida dos dados, então só pode ser calculada no treino. É exatamente a distinção que discutimos: "AMPLA" é constante (pode ficar fora), mediana é estatística (tem que ficar dentro).

2. RobustScaler (e não StandardScaler):
- Centraliza pela mediana e escala pelo IQR (intervalo interquartil) → robusto a outliers, coerente com a escolha da mediana.
- Por que normalizar? KNN (distâncias) e SVM (margem) são sensíveis à escala: sem normalizar, CLASSIFICACAO (vai a milhares) dominaria IDADE (dezenas). Para árvores é inofensivo (elas só comparam limiares). Então uma única escala serve aos 6 sem
prejudicar ninguém.

3. OneHotEncoder(handle_unknown="ignore"):
- One-Hot (e não Ordinal) porque as categóricas são nominais — TURNO, GRAU, SEXO não têm ordem. Ordinal inventaria uma hierarquia falsa (ex.: Noturno > Matutino) que enganaria os modelos numéricos.
- handle_unknown="ignore" → se aparecer no teste uma categoria que não estava no treino, ela vira tudo-zero em vez de quebrar. Blindagem contra o desconhecido.
- sparse_output=False → devolve matriz densa (numpy), compatível com todos os estimadores e com o GaussianNB.

4. Por que um único preprocessor para os 6 modelos?
- Comparação justa: todos recebem exatamente os mesmos dados transformados; qualquer diferença de desempenho vem do modelo, não de um pré-processamento diferente.
- No notebook de Modelagem, esse preprocessor entra como primeiro passo de um Pipeline por modelo → o fit reaprende imputação/escala só na dobra de treino durante a busca, fechando o anti-vazamento.

### Colocações

"Esse fit_transform na célula 16 não é vazamento? Você ajustou no treino balanceado inteiro."
  → Aqui é só demonstração do shape — esse objeto transformado (Xt) não é usado para treinar nada. No notebook de Modelagem, o preprocessor é re-ajustado dentro de cada Pipeline, só na partição de treino de cada passo. Vale deixar isso claro na defesa.

"Por que RobustScaler e não StandardScaler/MinMaxScaler?"
  → Pela presença de outliers nas notas e na MARGEM. StandardScaler usa média/desvio (sensíveis a extremos); MinMaxScaler é ainda pior (um único outlier comprime todo o resto). RobustScaler (mediana/IQR) é o mais estável aqui.

"O One-Hot não explode a dimensionalidade?"
  → Não, porque selecionamos categóricas de baixa cardinalidade (32 colunas no total). Foi por isso que descartamos curso/IES/município (alta cardinalidade) lá na seleção de features — eles sim explodiriam.

"Árvores precisam de normalização?"
  → Não, são invariantes a transformações monotônicas. Mas aplicar é inofensivo para elas, e usar um pipeline único simplifica e garante justiça na comparação. O custo é zero.

In [10]:
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scaler", RobustScaler())])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
    ("bin", "passthrough", bin_features),
])

# Demonstração da dimensionalidade resultante (ajuste só no treino balanceado)
Xt = preprocessor.fit_transform(X_train_bal)
print("Dimensão após pré-processamento:", Xt.shape)
print("Nº de colunas finais:", Xt.shape[1], "(6 numéricas + One-Hot das categóricas + MESMA_UF)")

Dimensão após pré-processamento: (20330, 32)
Nº de colunas finais: 32 (6 numéricas + One-Hot das categóricas + MESMA_UF)


### 2.6 Exportação das bases tratadas (entrega)

Salvamos as bases já tratadas para o pacote de entrega (link do Drive no `dbs_TP02.txt`):
a **base consolidada** (população do Problema B com alvo e features de engenharia, antes do split), os **três conjuntos** do Holdout estratificado 60/20/20 e o **treino balanceado** (undersampling) efetivamente usado no `fit`. CSV em UTF-8, separador `;`.

In [11]:
import os
DIR_BASES = "bases_tratadas"
os.makedirs(DIR_BASES, exist_ok=True)

def _join(Xpart, ypart):
    out = Xpart.copy()
    out["EFETIVOU"] = ypart.values
    return out

# Base consolidada: população do Problema B tratada (features + alvo), antes do split
_join(X, y).to_csv(f"{DIR_BASES}/base_tratada_consolidada_Grupo06.csv", sep=";", index=False, encoding="utf-8")
# Conjuntos do Holdout estratificado (mesma divisão de random_state=42)
_join(X_train, y_train).to_csv(f"{DIR_BASES}/treino_Grupo06.csv",    sep=";", index=False, encoding="utf-8")
_join(X_val,   y_val ).to_csv(f"{DIR_BASES}/validacao_Grupo06.csv",  sep=";", index=False, encoding="utf-8")
_join(X_test,  y_test).to_csv(f"{DIR_BASES}/teste_Grupo06.csv",      sep=";", index=False, encoding="utf-8")
# Treino balanceado (undersampling) usado no fit dos modelos
_join(X_train_bal, y_train_bal).to_csv(f"{DIR_BASES}/treino_balanceado_Grupo06.csv", sep=";", index=False, encoding="utf-8")

print("Bases tratadas exportadas em ./bases_tratadas/:")
for f in sorted(os.listdir(DIR_BASES)):
    print(f"  {f:<42} {os.path.getsize(os.path.join(DIR_BASES, f)):>9,} bytes")

Bases tratadas exportadas em ./bases_tratadas/:
  base_tratada_consolidada_Grupo06.csv       3,324,231 bytes
  teste_Grupo06.csv                            664,939 bytes
  treino_Grupo06.csv                         1,994,172 bytes
  treino_balanceado_Grupo06.csv              1,517,737 bytes
  validacao_Grupo06.csv                        665,410 bytes
